# Phase 4: Full Evaluation

**Runtime:** A100 GPU (Colab Pro/Pro+)

Comprehensive evaluation: all 5 DSL policies + baselines across
multiple workloads. Generates paper tables and figures.

**Prerequisites:**
- `01_trace_recording.ipynb` (traces on Drive)
- `02_profile_dispatch.ipynb` (dispatch overhead data)
- `03_baselines.ipynb` (baseline metrics)

In [ ]:
!pip install -q torch transformers accelerate lark>=1.1 matplotlib pandas

from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, time
import numpy as np
import matplotlib.pyplot as plt

DRIVE_ROOT = '/content/drive/MyDrive/moe-policylang-paper'
os.makedirs(f'{DRIVE_ROOT}/results/figures', exist_ok=True)

# Unzip moe_policylang from Google Drive
!unzip -qo {DRIVE_ROOT}/moe_policylang.zip -d /content/
sys.path.insert(0, '/content')

from moe_policylang import __version__
print(f'MoE-PolicyLang {__version__} imported successfully')

## 1. Load All Traces

In [ ]:
traces = {}
for fname in os.listdir(f'{DRIVE_ROOT}/traces'):
    if fname.endswith('.jsonl'):
        with open(f'{DRIVE_ROOT}/traces/{fname}') as f:
            header = json.loads(f.readline())
            data = [json.loads(line) for line in f]
        name = fname.replace('.jsonl', '')
        traces[name] = {'header': header, 'data': data}
        print(f'{name}: {len(data)} entries ({header["model_name"]})')

print(f'\nLoaded {len(traces)} traces')

## 2. Run All Policies on All Traces

In [ ]:
from moe_policylang.benchmark.policies import get_dsl_policies
from moe_policylang.runtime.hooks import build_hook

all_results = {}  # {trace_name: {policy_name: {metrics}}}

for trace_name, trace in traces.items():
    all_results[trace_name] = {}
    data = trace['data']
    print(f'\n=== {trace_name} ({len(data)} entries) ===')

    for policy_name, compiled in get_dsl_policies().items():
        hook = build_hook(compiled)
        latencies = []

        for entry in data:
            t0 = time.perf_counter()
            hook.on_layer(
                layer_idx=entry['l'],
                selected_experts=entry['e'],
                scores=entry.get('s'),
            )
            latencies.append(time.perf_counter() - t0)

        stats = hook.stats_snapshot()
        hits = stats['cache']['hits']
        misses = stats['cache']['misses']
        hr = hits / max(1, hits + misses)
        lat_us = np.array(latencies) * 1e6

        all_results[trace_name][policy_name] = {
            'hit_rate': hr,
            'hits': hits,
            'misses': misses,
            'evictions': stats['cache']['evictions'],
            'dispatch_mean_us': float(np.mean(lat_us)),
            'dispatch_p99_us': float(np.percentile(lat_us, 99)),
        }
        print(f'  {policy_name:<25} hit_rate={hr:.1%}  dispatch={np.mean(lat_us):.1f}µs')

## 3. Generate Paper Tables

In [ ]:
import pandas as pd

# Table 1: Hit rate matrix
rows = []
for trace_name in all_results:
    for policy_name, metrics in all_results[trace_name].items():
        rows.append({
            'Trace': trace_name,
            'Policy': policy_name,
            'Hit Rate': f"{metrics['hit_rate']:.1%}",
            'Dispatch (µs)': f"{metrics['dispatch_mean_us']:.1f}",
        })

df = pd.DataFrame(rows)
print('=== Table 1: Policy × Trace Hit Rates ===')
print(df.pivot(index='Policy', columns='Trace', values='Hit Rate').to_string())
print()
print('=== Table 2: Dispatch Overhead ===')
print(df.pivot(index='Policy', columns='Trace', values='Dispatch (µs)').to_string())

## 4. Generate Paper Figures

In [ ]:
# Figure 1: Rolling cache hit rate over time (on longest trace)
longest_trace = max(traces.keys(), key=lambda k: len(traces[k]['data']))
data = traces[longest_trace]['data']

fig, ax = plt.subplots(figsize=(10, 4))
window = 200

for policy_name, compiled in get_dsl_policies().items():
    hook = build_hook(compiled)
    hits_timeline = []
    for entry in data:
        plan = hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'], scores=entry.get('s'))
        s = hook.stats_snapshot()
        h, m = s['cache']['hits'], s['cache']['misses']
        hits_timeline.append(h / max(1, h + m))

    # Rolling average
    rolling = np.convolve(hits_timeline, np.ones(window)/window, mode='valid')
    ax.plot(rolling, label=policy_name, alpha=0.8)

ax.set_xlabel('Dispatch #')
ax.set_ylabel('Cumulative Hit Rate')
ax.set_title(f'Rolling Cache Hit Rate ({longest_trace})')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results/figures/fig1_rolling_hitrate.png', dpi=150)
plt.show()
print('Saved Figure 1')

In [ ]:
# Figure 2: Hit rate vs cache capacity (sweep)
from moe_policylang import MoEPolicyLang
from moe_policylang.compiler import compile_policy

capacities = [2, 4, 8, 16, 32]
trace_name = longest_trace
data = traces[trace_name]['data']

fig, ax = plt.subplots(figsize=(8, 4))

for eviction in ['lru', 'lfu']:
    hrs = []
    for cap in capacities:
        sched = MoEPolicyLang()
        kw = dict(capacity=cap, eviction=eviction)
        if eviction == 'lfu':
            kw['lfu_decay'] = 0.9
        ir = (
            sched.build(f'{eviction}_{cap}')
            .cache(**kw)
            .prefetch(strategy='none')
            .schedule(mode='gpu_only')
            .done()
        )
        hook = build_hook(compile_policy(ir))
        for entry in data:
            hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'])
        s = hook.stats_snapshot()
        h, m = s['cache']['hits'], s['cache']['misses']
        hrs.append(h / max(1, h + m))
    ax.plot(capacities, hrs, 'o-', label=eviction.upper())

ax.set_xlabel('Cache Capacity (# experts)')
ax.set_ylabel('Hit Rate')
ax.set_title(f'Hit Rate vs Cache Capacity ({trace_name})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results/figures/fig2_capacity_sweep.png', dpi=150)
plt.show()
print('Saved Figure 2')

In [ ]:
# Figure 3: Ablation — contribution of each composition axis
# cache-only → +prefetch → +triggers → full

data = traces[longest_trace]['data']
ablation_configs = [
    ('cache only',     dict(capacity=8, eviction='lfu', lfu_decay=0.9), dict(strategy='none'),              dict(mode='gpu_only'),  {}),
    ('+ prefetch',     dict(capacity=8, eviction='lfu', lfu_decay=0.9), dict(strategy='history', budget=4),  dict(mode='gpu_only'),  {}),
    ('+ scheduler',    dict(capacity=8, eviction='lfu', lfu_decay=0.9), dict(strategy='history', budget=4),  dict(mode='hybrid'),    {}),
    ('+ triggers',     dict(capacity=8, eviction='lfu', lfu_decay=0.9, ttl=50), dict(strategy='history', budget=4), dict(mode='hybrid'), {}),
]

ablation_results = []
for label, cache_kw, pf_kw, sched_kw, _extra in ablation_configs:
    sched = MoEPolicyLang()

    @sched.policy
    def ablation_policy(p, _ck=cache_kw, _pk=pf_kw, _sk=sched_kw):
        p.cache(**_ck)
        p.prefetch(**_pk)
        p.schedule(**_sk)

    ir = sched.policies['ablation_policy']
    hook = build_hook(compile_policy(ir))
    for entry in data:
        hook.on_layer(layer_idx=entry['l'], selected_experts=entry['e'])
    s = hook.stats_snapshot()
    h, m = s['cache']['hits'], s['cache']['misses']
    hr = h / max(1, h + m)
    ablation_results.append((label, hr))
    print(f'{label:<20} hit_rate={hr:.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
labels = [r[0] for r in ablation_results]
hrs = [r[1] for r in ablation_results]
ax.bar(labels, hrs, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
ax.set_ylabel('Hit Rate')
ax.set_title('Ablation: Contribution of Each Composition Axis')
for i, v in enumerate(hrs):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(f'{DRIVE_ROOT}/results/figures/fig3_ablation.png', dpi=150)
plt.show()
print('Saved Figure 3')

## 5. Save All Results

In [ ]:
# Combine with baseline results
baseline_path = f'{DRIVE_ROOT}/results/baseline_results.json'
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baselines = json.load(f)
else:
    baselines = {}

full_results = {
    'moe_policylang_policies': all_results,
    'baselines': baselines,
    'ablation': {r[0]: r[1] for r in ablation_results},
}

out_path = f'{DRIVE_ROOT}/results/full_evaluation.json'
with open(out_path, 'w') as f:
    json.dump(full_results, f, indent=2, default=str)
print(f'Saved to {out_path}')